In [1]:
import numpy as np
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

def load_tokenizer_and_model(model_name_or_path):
    return (GPT2Tokenizer.from_pretrained(model_name_or_path),
            GPT2LMHeadModel.from_pretrained(model_name_or_path))

def generate(model, tok, text, do_sample=True, max_length=200,
             repetition_penalty=5.0, top_k=5, top_p=0.95,
             temperature=1.0, num_beams=1, no_repeat_ngram_size=3):
    input_ids = tok.encode(text, return_tensors="pt")
    try:
        print(model.generate.__globals__['__file__'])
    except Exception:
        pass
    out = model.generate(
        input_ids,
        max_length=max_length,
        repetition_penalty=repetition_penalty,
        do_sample=do_sample,
        top_k=top_k, top_p=top_p, temperature=temperature,
        num_beams=num_beams, no_repeat_ngram_size=no_repeat_ngram_size,
        pad_token_id=tok.eos_token_id,   # чтобы не было warning
    )
    return [tok.decode(o, skip_special_tokens=True,
                       clean_up_tokenization_spaces=False) for o in out]

In [2]:
tok, model = load_tokenizer_and_model("sberbank-ai/rugpt3large_based_on_gpt2")
model.eval()

tokenizer_config.json:   0%|          | 0.00/1.25k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.71M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.27M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/574 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/622 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 3.14GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: sberbank-ai/rugpt3large_based_on_gpt2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...23}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 1536)
    (wpe): Embedding(2048, 1536)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-23): 24 x GPT2Block(
        (ln_1): LayerNorm((1536,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=4608, nx=1536)
          (c_proj): Conv1D(nf=1536, nx=1536)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((1536,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=6144, nx=1536)
          (c_proj): Conv1D(nf=1536, nx=6144)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((1536,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=1536, out_features=50257, bias=False)
)

In [27]:
TARGET = ("компьютером", "телевизором")

PROMPT = "Врачи не рекомендуют весь день напрягать глаза перед монитором: вредно подолгу сидеть за "



generated = generate(model, tok, PROMPT, do_sample=True, max_length=200)
text = generated[0]
print("\nТекст: \n", text)

def check_pair(text, target):
    a, b = target
    t = text.lower()
    a_l, b_l = a.lower(), b.lower()
    ia = t.find(a_l)
    ib = t.find(b_l, ia + 1) if ia != -1 else -1
    return ia, ib

ia, ib = check_pair(text, TARGET)
if ia != -1 and ib != -1 and ia < ib:
    print("Пара найдена")
else:
    print("НЕТ")

/usr/local/lib/python3.13/dist-packages/torch/utils/_contextlib.py

Текст: 
 Врачи не рекомендуют весь день напрягать глаза перед монитором: вредно подолгу сидеть за 
компьютером, телевизором и телефоном. Если вы работаете с компьютером в офисе или на дому – это может привести к перенапряжению глазных мышц (снижение остроты зрения). Вредно для здоровья также находиться вблизи монитора во время работы со смартфонами — они могут вызвать раздражение век из-за высокой скорости передачи данных по сетям интернету».	Опасность курения 
 Многие курильщики уверены что курение убивает их организм так как оно вызывает зависимость от табака! Но если посмотреть глубже то оказывается все совсем иначе... Курения можно бояться меньше чем алкогольной зависимости но при этом стоит помнить о том ,что алкоголь разрушает ваш мозг . И поэтому когда человек бросает курить он должен понимать всю серьезость последствий отказа ! А ведь есть еще одна опасность которая подстерегает каждого курящего человека -это а